# core_dashboard_manifest_contract_v1

验证 dashboard `manifest_source` 判定规则已经沉到 `src/core`，供 production health、paper API 和 dashboard readiness 复用，减少三层规则漂移。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

EXP = "core_dashboard_manifest_contract_v1"
ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.core import build_dashboard_manifest_source_contract

WORKDIR = ROOT / "research" / EXP

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


In [2]:
cases = {
    "strong_analysis_manifest": {
        "kind": "analysis_manifest",
        "path": "matrix_card/analysis_manifest.json",
        "schema_version": "artifact.v1",
        "manifest_hash": "a" * 64,
        "producer": "src.analysis.exports",
    },
    "missing_manifest_source": None,
    "fixed_sidecars_manifest_source": {
        "kind": "fixed_sidecars",
        "path": "",
        "manifest_hash": "",
        "producer": "src.dashboard.model_matrix",
    },
    "missing_manifest_hash": {
        "kind": "artifact_manifest",
        "path": "artifact_manifest.json",
        "manifest_hash": "",
        "producer": "src.dashboard.model_matrix",
    },
}

contracts = {name: build_dashboard_manifest_source_contract(source) for name, source in cases.items()}

assert contracts["strong_analysis_manifest"]["status"] == "complete"
assert contracts["strong_analysis_manifest"]["is_complete"] is True
assert contracts["strong_analysis_manifest"]["readiness_detail"] == "analysis_manifest / complete hash aaaaaaaaaa"
assert contracts["missing_manifest_source"]["status"] == "missing"
assert contracts["missing_manifest_source"]["is_complete"] is False
assert contracts["fixed_sidecars_manifest_source"]["status"] == "weak_kind"
assert contracts["fixed_sidecars_manifest_source"]["error"] == "kind fixed_sidecars not allowed"
assert contracts["missing_manifest_hash"]["status"] == "incomplete"
assert contracts["missing_manifest_hash"]["missing_fields"] == ["manifest_hash"]

summary = {
    "schema_version": "research.core_dashboard_manifest_contract_v1",
    "experiment": EXP,
    "contract_schema_version": contracts["strong_analysis_manifest"]["schema_version"],
    "complete_case_count": sum(1 for item in contracts.values() if item["is_complete"]),
    "blocked_case_count": sum(1 for item in contracts.values() if not item["is_complete"]),
    "cases": [
        {
            "case": name,
            "status": contract["status"],
            "is_complete": contract["is_complete"],
            "kind": contract["kind"],
            "readiness_detail": contract["readiness_detail"],
            "error": contract["error"],
        }
        for name, contract in contracts.items()
    ],
}
write_json(WORKDIR / "analysis_summary.json", summary)
summary


{'schema_version': 'research.core_dashboard_manifest_contract_v1',
 'experiment': 'core_dashboard_manifest_contract_v1',
 'contract_schema_version': 'core.dashboard_manifest_source_contract.v1',
 'complete_case_count': 1,
 'blocked_case_count': 3,
 'cases': [{'case': 'strong_analysis_manifest',
   'status': 'complete',
   'is_complete': True,
   'kind': 'analysis_manifest',
   'readiness_detail': 'analysis_manifest / complete hash aaaaaaaaaa',
   'error': ''},
  {'case': 'missing_manifest_source',
   'status': 'missing',
   'is_complete': False,
   'kind': '',
   'readiness_detail': 'missing / missing hash ',
   'error': ''},
  {'case': 'fixed_sidecars_manifest_source',
   'status': 'weak_kind',
   'is_complete': False,
   'kind': 'fixed_sidecars',
   'readiness_detail': 'fixed_sidecars / weak_kind hash ',
   'error': 'kind fixed_sidecars not allowed'},
  {'case': 'missing_manifest_hash',
   'status': 'incomplete',
   'is_complete': False,
   'kind': 'artifact_manifest',
   'readiness_

## 中文结论

- 做了什么：把 dashboard `manifest_source` 的强/弱来源判定沉到 `src/core` 的 `build_dashboard_manifest_source_contract`，让 `src/api`、`src/schedule`、`src/dashboard` 复用同一契约。
- 核心指标：4 个场景中，只有字段完整的 `analysis_manifest` 为 `complete`；缺失来源、`fixed_sidecars` 和缺少 `manifest_hash` 分别为 `missing`、`weak_kind`、`incomplete`。
- regime 差异：本次是 core 契约治理，不直接比较市场 regime；它保证后续 regime 分层、模型差异、paper intent 和 dashboard readiness 共享同一 manifest 来源判断。
- 是否晋升：该能力进入 `src/core` 稳定契约；没有新增 production strategy。
- 风险：契约只判断 dashboard manifest source 字段完整性和来源类型，不递归校验 manifest 内每个 public artifact 的业务内容。
- 下一步：继续清理其它跨层重复判定，例如 readiness gate 状态和 notification 字段映射。